**install required libraries**

In [118]:
!pip install faiss-cpu sentence-transformers haversine  spacy en_core_web_sm fuzzywuzzy


In [119]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from fuzzywuzzy import fuzz


In [120]:
##Route and Station names with Latitude and Longitude
csv_file = 'https://dagshub.com/Omdena/VITBhopalUniversity_ChatbotforBRTSNavigation/raw/99c2e8d2883dd9faaa68ed60d5405dd40e77c456/src/tasks/task-2/Routes/all_routes_combined.csv'

df = pd.read_csv(csv_file)
df.head()

,Station,Latitude,Longitude,Route,link
0,Halalpur Bus Stand,23.273266,77.363991,SR1,"https://www.google.com/maps/place/23.27326584,..."
1,Lalghati,23.273069,77.368469,SR1,"https://www.google.com/maps/place/23.27306938,..."
2,Vip Guest,23.271318,77.375740,SR1,"https://www.google.com/maps/place/23.27131844,..."
3,Koh-E-Fiza,23.269117,77.377274,SR1,"https://www.google.com/maps/place/23.26911736,..."
4,Collectorate,23.263588,77.383125,SR1,"https://www.google.com/maps/place/23.26358795,..."


In [121]:
# Load FAQ static data  file for  questions and answers we will train this data in the RAG pipeline later
excel_file = '/content/Chatbot Dataset.xlsx'
general_info=pd.read_excel(excel_file)

In [122]:
general_info.head()

,Question,Answer
0,General Information,NaN
1,What is BRTS System,Bhopal BRTS was a bus rapid transit system loc...
2,How does the BRTS system work?,Bhopal's Bus Rapid Transit System (BRTS) is de...
3,What are the main features of Bhopal's BRTS?,"Bhopal’s Bus Rapid Transit System (BRTS), know..."
4,What are the benefits of using the BRTS system?,The BRTS (Bus Rapid Transit System) offers num...


##Generate embeedings
will create for station names for faster retrevial --using **SentenceTransformer** Model


In [123]:
# Initialize the model
def initialize_model(model_name):
    return SentenceTransformer(model_name)

##Create Embeedings for Station names and General questions
def create_embeddings(model,data):
    return np.array(model.encode(data.tolist()))




In [124]:
model = initialize_model('paraphrase-MiniLM-L6-v2')
question_embeddings = create_embeddings(model, general_info['Question'])
station_embeddings = create_embeddings(model, df['Station'])

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


FAISS Index Initialization:
FAISS uses an index for fast vector search



In [125]:
import faiss
def initialize_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

faiss_general_info_index = initialize_faiss_index(question_embeddings)
faiss_station_index = initialize_faiss_index(station_embeddings)

# Function to retrieve the closest matching question


In [126]:
def retrieve_general_info(query):
    query_embedding = model.encode([query])
    D, I = faiss_general_info_index.search(np.array(query_embedding), k=1)

    if D[0][0] < 0.5:
        answer = general_info['Answer'].iloc[I[0][0]]
        return answer
    else:
        return None

In [127]:
def retrieve_top_k_matches(query, model, faiss_index, general_info, k=3):
    query_embedding = model.encode([query])
    D, I = faiss_general_info_index.search(np.array(query_embedding), k=k)
    top_matches = [(general_info['Question'].iloc[idx], general_info['Answer'].iloc[idx]) for idx in I[0]]
    return top_matches

In [128]:
def retrieve_top_k_matches(query, model, faiss_index, general_info, k=3):
    query_embedding = model.encode([query])
    D, I = faiss_index.search(np.array(query_embedding), k=k)
    top_matches = [(D[0][i], general_info['Question'].iloc[I[0][i]], general_info['Answer'].iloc[I[0][i]]) for i in range(k)]
    return top_matches

`GPT-2 as LLM Generator`

In [129]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
generator_model = GPT2LMHeadModel.from_pretrained("gpt2")

# Set pad_token
tokenizer.pad_token = tokenizer.eos_token


def generate_answer_with_context(query, retrieved_context):
    if retrieved_context:
        context = f"Q: {retrieved_context[0][0]}\nA: {retrieved_context[0][1]}"
    else:
        context = ""

    prompt = f"{context}\nQ: {query}\nA:"

    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

    outputs = generator_model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=50,
        pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_start = generated_text.find("A:") + 2
    return generated_text[answer_start:].strip()




In [130]:
def retrieve_or_generate_answer(query, model, faiss_index, general_info, k=1, threshold=0.5):
    top_matches = retrieve_top_k_matches(query, model, faiss_index, general_info, k=k)

    if top_matches and top_matches[0][0] < threshold:
        return f"Q: {query}\nA: {top_matches[0][1]}"

    retrieved_context = [(q, a) for _, q, a in top_matches]
    generated_answer = generate_answer_with_context(query, retrieved_context)
    return f"Q: {query}\nA: {generated_answer}"

In [131]:
def get_available_stations():
    return ", ".join(df['Station'].unique())

In [132]:
import spacy
from fuzzywuzzy import process
import nltk
nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")
stop_words = set(nlp.Defaults.stop_words)

# Function to extract station names using exact or fuzzy matching
def extract_stations_from_query(query):
    doc = nlp(query)
    station_names = df['Station'].str.lower().tolist()
    extracted_stations = []

    for chunk in doc.noun_chunks:
        chunk_text = chunk.text.lower()
        if chunk_text in station_names:
            extracted_stations.append(chunk.text)
        elif chunk_text not in stop_words:
            match, score = process.extractOne(chunk_text, station_names)
           # print(f"Checking chunk: '{chunk.text}', Match: '{match}', Score: {score}")  # Debugging output
            if score > 90:  # threshold
                extracted_stations.append(match)

    # Remove duplicates
    extracted_stations = list(dict.fromkeys(extracted_stations))

    return extracted_stations

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [133]:
from haversine import haversine, Unit

# Function to calculate distance and travel time
def calculate_distance_and_time(station_a, station_b):

    # Convert both station_a and station_b to lowercase for comparison
    station_a_lower = station_a.lower()
    station_b_lower = station_b.lower()

    # Convert the 'Station' column in the DataFrame to lowercase for matching
    station_names_lower = df['Station'].str.lower()

    # Check if both stations are present in the DataFrame (case-insensitive)
    if station_a_lower not in station_names_lower.values:
        return None, None, f"Station '{station_a}' not found."

    if station_b_lower not in station_names_lower.values:
        return None, None, f"Station '{station_b}' not found."

    # Retrieve the exact row for each station (using the original DataFrame)
    row_a = df[station_names_lower == station_a_lower].iloc[0]
    row_b = df[station_names_lower == station_b_lower].iloc[0]

    coord_a = (row_a['Latitude'], row_a['Longitude'])
    coord_b = (row_b['Latitude'], row_b['Longitude'])

    distance = haversine(coord_a, coord_b, unit=Unit.KILOMETERS)

    average_speed_kmph = 40  # Example average speed
    travel_time = distance / average_speed_kmph * 60  # Convert hours to minutes

    return distance, travel_time, None  # No error message


In [134]:
# Main response generation function
def generate_response(query, retrieved_data=None, general_info=None):
    # Extract stations from the user's query
    stations = extract_stations_from_query(query)

    if len(stations) < 2:
        available_stations = get_available_stations()
        return f"Please provide two valid station names for the calculation. The available stations are: {available_stations}"


    station_a, station_b = stations[0], stations[1]

    # Calculate distance and travel time
    distance, travel_time, error_message = calculate_distance_and_time(station_a, station_b)

    if error_message is not None:
        return error_message  # Return the error if a station wasn't found

    return f"The distance from {station_a} to {station_b} is {distance:.2f} km, and it will take approximately {travel_time:.2f} minutes."


In [135]:

# Initialize the sentence transformer model
model = initialize_model('paraphrase-MiniLM-L6-v2')

# Create embeddings for general information questions and initialize FAISS index
station_embeddings = create_embeddings(model, df['Station'])
faiss_station_index = initialize_faiss_index(station_embeddings)

test_query = "how soon i can reach from Moti Masjid and  halalpur Bus Stand"
response = generate_response(test_query)
print(response)

The distance from Moti Masjid to halalpur Bus Stand is 4.02 km, and it will take approximately 6.04 minutes.


In [136]:
# Initialize the sentence transformer model
model = initialize_model('paraphrase-MiniLM-L6-v2')

question_embeddings = create_embeddings(model, general_info['Question'])
faiss_general_info_index = initialize_faiss_index(question_embeddings)

query = "Tell me how the BRTS system works."

answer = retrieve_or_generate_answer(query, model, faiss_general_info_index, general_info)
print(answer)


/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


Q: Tell me how the BRTS system works.
A: Bhopal's Bus Rapid Transit System (BRTS) is designed to enhance urban mobility by providing a fast, reliable, and eco-friendly public transport solution. Operating on dedicated bus lanes, the BRTS minimizes travel delays caused by traffic congestion, ensuring that passengers can commute efficiently across the city. With strategically placed bus stations featuring real-time information displays, pre-boarding ticketing, and comfortable waiting areas, the system prioritizes user convenience. Bhopal's BRTS also integrates with other modes of transport, allowing seamless transfers for passengers. Employing high-frequency services and environmentally friendly vehicles, such as electric and hybrid buses, the BRTS not only addresses the pressing need for efficient public transport but also contributes to reducing urban pollution. Designed with accessibility in mind, the BRTS is committed to serving all residents, making it a vital component of Bhopal's 

In [137]:
def retrieve_top_k_matches(query, model, faiss_index, info, k=1):
    query_embedding = model.encode([query])
    distances, indices = faiss_index.search(query_embedding, k)
    results = [(distances[0][i], info['Question'][indices[0][i]], info['Answer'][indices[0][i]]) for i in range(k)]
    return results


In [138]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
generator_model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def generate_answer_with_context(query, retrieved_context):
    if retrieved_context:
        context = "\n".join([f"Fact: {a}" for q, a in retrieved_context])
    else:
        context = "No relevant information was found."

    # Adjusted prompt to guide GPT-2's response generation
    # Create a structured prompt
    prompt = (
        "Use ONLY the following information to answer the question. "
        "If the information is not sufficient, say so.\n\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        "Answer: "
    )
    print("Prompt for GPT-2:", prompt)  # Debugging output
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    outputs = generator_model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=30,  # Limit to prevent hallucination
        pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Post-process to clean up the answer based on prompt
    answer_start = generated_text.find("A:") + 2
    answer = generated_text[answer_start:].strip()

    return answer if answer else "I'm sorry, I couldn't generate a relevant answer based on the information available."





In [139]:
def retrieve_or_generate_answer(query, model, faiss_general_info_index, general_info, faiss_station_index, k=1, threshold=0.5):
    # Retrieve general info matches and filter relevant info only
    general_matches = retrieve_top_k_matches(query, model, faiss_general_info_index, general_info, k=k)
    print("General Matches:", general_matches)  # Debugging output

    # Check if station info is needed and retrieve if both stations are specified
    stations = extract_stations_from_query(query)
    print("Extracted Stations:", stations)  # Debugging output

    station_retrieved_data = []
    if len(stations) == 2:
        station_a, station_b = stations[0], stations[1]
        distance, travel_time, error_message = calculate_distance_and_time(station_a, station_b)
        if not error_message:
            station_retrieved_data = [(query, f"The distance from {station_a} to {station_b} is {distance:.2f} km, and it will take approximately {travel_time:.2f} minutes.")]

    # Selectively include only relevant BRTS general info and station-specific info
    relevant_general_info = [(q, a) for _, q, a in general_matches if _ < threshold and 'BRTS' in a]

    # Combine the general BRTS context with specific station information if found
    retrieved_context = relevant_general_info + station_retrieved_data
    if retrieved_context:
        return generate_answer_with_context(query, retrieved_context)
    else:
        return "I'm sorry, I couldn't find relevant information for your query."


# Sample usage
query = "Tell me how the BRTS system works. How soon can I reach from Moti Masjid to VIP Guest"
answer = retrieve_or_generate_answer(query, model, faiss_general_info_index, general_info, faiss_station_index)
print(answer)


General Matches: [(38.32589, 'How does the BRTS system work?', "Bhopal's Bus Rapid Transit System (BRTS) is designed to enhance urban mobility by providing a fast, reliable, and eco-friendly public transport solution. Operating on dedicated bus lanes, the BRTS minimizes travel delays caused by traffic congestion, ensuring that passengers can commute efficiently across the city. With strategically placed bus stations featuring real-time information displays, pre-boarding ticketing, and comfortable waiting areas, the system prioritizes user convenience. Bhopal's BRTS also integrates with other modes of transport, allowing seamless transfers for passengers. Employing high-frequency services and environmentally friendly vehicles, such as electric and hybrid buses, the BRTS not only addresses the pressing need for efficient public transport but also contributes to reducing urban pollution. Designed with accessibility in mind, the BRTS is committed to serving all residents, making it a vital